# **Solana Validator Behavior Analysis**

## **1) Load Initial Dataset**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.cloud import bigquery

client = bigquery.Client()

# Load selected features from vote account rows
query = """
SELECT
    node_pubkey,
    pubkey,
    block_timestamp,
    commission,
    lamports,
    root_slot,
    epoch_credits,
    votes,
    last_timestamp,
    authorized_voters,
    prior_voters,
    authorized_withdrawer,
    tx_signature,
    retrieval_timestamp,
    block_slot,
    block_hash,
    owner,
    account_type,
    rent_epoch,
    space
FROM `bigquery-public-data.crypto_solana_mainnet_us.Accounts`
WHERE account_type = 'vote'
"""

validators = client.query(query).to_dataframe()

validators.head()

In [ ]:
# Sanity Check
print("Shape:", validators.shape)
print("\nColumns:")
print(validators.columns.tolist())

validators.info()

## **2) Preliminary EDA**

### **2.1) Missingness**

In [ ]:
missing = pd.DataFrame({
    "missing_count": validators.isna().sum(),
    "missing_pct": validators.isna().mean() * 100
}).sort_values("missing_pct", ascending=False)

missing["non_missing_pct"] = 100 - missing["missing_pct"]

missing

In [ ]:
# Deduplicate Validators (keep most recent snapshot per node)

print("Before deduplication:")
print(f"Rows: {len(validators):,}")
print(f"Unique validators: {validators['node_pubkey'].nunique():,}")
print(f"Duplicate rows: {len(validators) - validators['node_pubkey'].nunique():,}")

validators = (
    validators
    .sort_values("retrieval_timestamp")
    .drop_duplicates(
        subset="node_pubkey",
        keep="last"
    )
    .reset_index(drop=True)
)

print("\nAfter deduplication:")
print(f"Rows: {len(validators):,}")
print(f"Unique validators: {validators['node_pubkey'].nunique():,}")
print(f"Duplicate rows: {len(validators) - validators['node_pubkey'].nunique():,}")

### **2.2) Distributions and Summary Statistics of Selected Features**

In [ ]:
# block_timestamp
validators["block_timestamp"] = pd.to_datetime(validators["block_timestamp"])

print(validators["block_timestamp"].describe())

validators["block_date"] = validators["block_timestamp"].dt.date

daily_counts = validators.groupby("block_date").size()

daily_counts.plot(figsize=(10, 4))
plt.title("Vote Account Records Over Time")
plt.xlabel("Date")
plt.ylabel("Number of Records")
plt.show()

In [ ]:
# node_pubkey
print(validators["node_pubkey"].value_counts().describe())
print(validators["node_pubkey"].nunique())
validators["node_pubkey"].value_counts().head(20)

In [ ]:
# commission
print(validators["commission"].describe())
print("\nMissing:", validators["commission"].isna().sum())

validators["commission"].hist(bins=30)
plt.title("Distribution of Validator Commission")
plt.xlabel("Commission")
plt.ylabel("Count")
plt.show()

In [ ]:
validators["commission"].value_counts().sort_index()

In [ ]:
# lamports
validators["lamports"] = pd.to_numeric(
    validators["lamports"],
    errors="coerce"
)

print(validators["lamports"].describe(

    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]

))

validators["lamports"].hist(bins=50)
plt.title("Distribution of Vote Account Lamports")
plt.xlabel("Lamports")
plt.ylabel("Count")
plt.show()

np.log1p(validators["lamports"].astype(float)).hist(bins=50)
plt.title("Log Distribution of Vote Account Lamports")
plt.xlabel("log(1 + lamports)")
plt.ylabel("Count")
plt.show()

In [ ]:
# root_slot
print(validators["root_slot"].describe())
print("\nMissing:", validators["root_slot"].isna().sum())

validators["root_slot"].hist(bins=50)
plt.title("Distribution of Root Slot")
plt.xlabel("Root Slot")
plt.ylabel("Count")
plt.show()

In [ ]:
# epoch_credits
validators["num_epoch_credit_records"] = validators["epoch_credits"].apply(
    lambda x: len(x) if isinstance(x, (list, np.ndarray)) else 0
)

print(validators["num_epoch_credit_records"].describe())
print("\nMissing:", validators["epoch_credits"].isna().sum())

validators["num_epoch_credit_records"].hist(bins=20)
plt.title("Number of Epoch Credit Records per Validator")
plt.xlabel("Number of Epoch Credit Records")
plt.ylabel("Count")
plt.show()

validators[["node_pubkey", "epoch_credits", "num_epoch_credit_records"]].head()

In [ ]:
# epoch_credits (nested data)
# Show full epoch credit records

for i in range(5):
    print(f"\nValidator {i}")
    print("Node:", validators.iloc[i]["node_pubkey"])
    print("Epoch Credits:")

    for record in validators.iloc[i]["epoch_credits"]:
        print(record)

    print("-" * 80)

In [ ]:
# votes
validators["num_vote_records"] = validators["votes"].apply(
    lambda x: len(x) if isinstance(x, (list, np.ndarray)) else 0
)

print(validators["num_vote_records"].describe())
print("\nMissing:", validators["votes"].isna().sum())

validators["num_vote_records"].hist(bins=30)
plt.title("Number of Vote Records per Validator")
plt.xlabel("Number of Vote Records")
plt.ylabel("Count")
plt.show()

validators[["node_pubkey", "votes", "num_vote_records"]].head()

In [ ]:
for i in range(5):
    print(f"\nValidator {i}")
    print("Node:", validators.iloc[i]["node_pubkey"])
    print("Votes:")

    votes = validators.iloc[i]["votes"]

    if len(votes) > 0:
        for vote in votes:
            print(vote)
    else:
        print("No votes")

    print("-" * 80)

In [ ]:
# last_timestamp
"""validators["num_last_timestamp_records"] = validators["last_timestamp"].apply(
    lambda x: len(x) if isinstance(x, (list, np.ndarray)) else 0
)

print(validators["num_last_timestamp_records"].describe())
print("\nMissing:", validators["last_timestamp"].isna().sum())

validators[["node_pubkey", "last_timestamp", "num_last_timestamp_records"]].head()"""

In [ ]:
# last_timestamp EDA

def extract_last_vote_slot(x):
    if x is None:
        return np.nan

    if isinstance(x, np.ndarray):
        x = x.tolist()

    if isinstance(x, list) and len(x) > 0:
        record = x[0]
        if isinstance(record, dict):
            return record.get("slot")

    return np.nan


def extract_last_vote_timestamp(x):
    if x is None:
        return pd.NaT

    if isinstance(x, np.ndarray):
        x = x.tolist()

    if isinstance(x, list) and len(x) > 0:
        record = x[0]
        if isinstance(record, dict):
            return pd.to_datetime(record.get("timestamp"), errors="coerce")

    return pd.NaT


validators["latest_vote_slot"] = validators["last_timestamp"].apply(extract_last_vote_slot)
validators["latest_vote_timestamp"] = validators["last_timestamp"].apply(extract_last_vote_timestamp)

validators["latest_vote_slot"] = pd.to_numeric(validators["latest_vote_slot"], errors="coerce")

print("Latest Vote Slot")
print(validators["latest_vote_slot"].describe())

print("\nLatest Vote Timestamp")
print(validators["latest_vote_timestamp"].describe())

print("\nMissing Slots:", validators["latest_vote_slot"].isna().sum())
print("Missing Timestamps:", validators["latest_vote_timestamp"].isna().sum())

validators[["node_pubkey", "last_timestamp", "latest_vote_slot", "latest_vote_timestamp"]].head()

# Distribution of latest vote slot

validators["latest_vote_slot"].hist(bins=50)

plt.title("Distribution of Latest Vote Slot")
plt.xlabel("Latest Vote Slot")
plt.ylabel("Count")
plt.show()


# Distribution of latest vote timestamp

validators["latest_vote_timestamp"] = pd.to_datetime(
    validators["latest_vote_timestamp"],
    errors="coerce"
)

validators["latest_vote_timestamp"].hist(bins=50)

plt.title("Distribution of Latest Vote Timestamp")
plt.xlabel("Timestamp")
plt.ylabel("Count")
plt.show()

In [ ]:
# authorized_voters
validators["num_authorized_voters"] = validators["authorized_voters"].apply(
    lambda x: len(x) if isinstance(x, (list, np.ndarray)) else 0
)

print(validators["num_authorized_voters"].describe())
print("\nMissing:", validators["authorized_voters"].isna().sum())

validators["num_authorized_voters"].hist(bins=20)
plt.title("Number of Authorized Voters per Vote Account")
plt.xlabel("Number of Authorized Voters")
plt.ylabel("Count")
plt.show()

validators[["node_pubkey", "authorized_voters", "num_authorized_voters"]].head()

In [ ]:
# prior_voters
validators["num_prior_voters"] = validators["prior_voters"].apply(
    lambda x: len(x) if isinstance(x, (list, np.ndarray)) else 0
)

print(validators["num_prior_voters"].describe())
print("\nMissing:", validators["prior_voters"].isna().sum())

validators["num_prior_voters"].hist(bins=20)
plt.title("Number of Prior Voters per Vote Account")
plt.xlabel("Number of Prior Voters")
plt.ylabel("Count")
plt.show()

validators[["node_pubkey", "prior_voters", "num_prior_voters"]].head()

In [ ]:
# rent_epoch
print(validators["rent_epoch"].describe())
print("\nMissing:", validators["rent_epoch"].isna().sum())

validators["rent_epoch"].hist(bins=30)
plt.title("Distribution of Rent Epoch")
plt.xlabel("Rent Epoch")
plt.ylabel("Count")
plt.show()

In [ ]:
# space
print(validators["space"].describe())
print("\nMissing:", validators["space"].isna().sum())

validators["space"].hist(bins=30)
plt.title("Distribution of Account Storage Space")
plt.xlabel("Space")
plt.ylabel("Count")
plt.show()

## **3) Preprocessing and Feature Engineering**

### **3.1) Feature Engineering and Validation**

In [ ]:
# Epoch Helper Function
MAX_EPOCH_CREDITS = 6912000

def get_epoch_deltas(epoch_credit_records):
    """
    Returns epoch credit gains for each recorded epoch.
    """

    if not isinstance(epoch_credit_records, (list, np.ndarray)):
        return []

    deltas = []

    for record in epoch_credit_records:
        try:
            credits = int(record["credits"])
            prev = int(record["previous_credits"])
            deltas.append(credits - prev)
        except:
            continue

    return deltas

In [ ]:
# Mean Efficiency
validators["mean_efficiency_pct"] = validators["epoch_credits"].apply(
    lambda x: (
        np.mean(get_epoch_deltas(x)) / MAX_EPOCH_CREDITS * 100
    )
    if len(get_epoch_deltas(x)) > 0
    else np.nan
)

validators["mean_efficiency_pct"].describe()

In [ ]:
# Median Efficiency
validators["median_efficiency_pct"] = validators["epoch_credits"].apply(
    lambda x: (
        np.median(get_epoch_deltas(x)) / MAX_EPOCH_CREDITS * 100
    )
    if len(get_epoch_deltas(x)) > 0
    else np.nan
)

validators["median_efficiency_pct"].describe()

In [ ]:
def efficiency_cv(epoch_credit_records):

    deltas = get_epoch_deltas(epoch_credit_records)

    if len(deltas) < 2:
        return np.nan

    mean = np.mean(deltas)

    if mean == 0:
        return np.nan

    return np.std(deltas) / mean


validators["efficiency_cv"] = validators["epoch_credits"].apply(
    efficiency_cv
)

validators["efficiency_cv"].describe()

In [ ]:
# Performance Trend
def performance_trend(epoch_credit_records):

    deltas = get_epoch_deltas(epoch_credit_records)

    if len(deltas) < 4:
        return np.nan

    efficiencies = np.array(deltas) / MAX_EPOCH_CREDITS * 100

    midpoint = len(efficiencies) // 2

    first_half = efficiencies[:midpoint]
    second_half = efficiencies[midpoint:]

    return np.mean(second_half) - np.mean(first_half)


validators["performance_trend"] = validators["epoch_credits"].apply(
    performance_trend
)

validators["performance_trend"].describe()

In [ ]:
# Number of Epochs
validators["num_epochs_recorded"] = validators["epoch_credits"].apply(
    lambda x: len(x)
    if isinstance(x, (list, np.ndarray))
    else 0
)

validators["num_epochs_recorded"].describe()

In [ ]:
# Log Lamports
validators["log_lamports"] = np.log10(
    validators["lamports"] + 1
)

validators["log_lamports"].describe()

In [ ]:
validators["num_authorized_voters"] = validators["authorized_voters"].apply(
    lambda x: len(x)
    if isinstance(x, (list, np.ndarray))
    else 0
)

validators["num_authorized_voters"].describe()

In [ ]:
validators["num_prior_voters"] = validators["prior_voters"].apply(
    lambda x: len(x)
    if isinstance(x, (list, np.ndarray))
    else 0
)

validators["num_prior_voters"].describe()

In [ ]:
# Establish Candidate Feature Set
candidate_features = [
    "commission",
    "log_lamports",
    "mean_efficiency_pct",
    "median_efficiency_pct",
    "efficiency_cv",
    "performance_trend",
    "num_epochs_recorded",
    "num_authorized_voters",
    "num_prior_voters"
]

validators[candidate_features].describe()

In [ ]:
# Plot Distributions of Candidate Set
import matplotlib.pyplot as plt

for feature in candidate_features:

    plt.figure(figsize=(6,4))

    validators[feature].hist(bins=40)

    plt.title(feature)
    plt.xlabel(feature)
    plt.ylabel("Count")

    plt.show()

In [ ]:
for feature in candidate_features:

    plt.figure(figsize=(6,1.5))

    validators.boxplot(column=feature, vert=False)

    plt.title(feature)

    plt.show()

In [ ]:
# Remove the single invalid commission observation
validators = validators[validators["commission"] <= 100].copy()

In [ ]:
# Remove Governance Features
candidate_features = [
    "commission",
    "log_lamports",
    "mean_efficiency_pct",
    "median_efficiency_pct",
    "efficiency_cv",
    "performance_trend",
    "num_epochs_recorded"
]

In [ ]:
print(validators[candidate_features].isna().sum())
print(validators[candidate_features].dropna().shape)

In [ ]:
# Handle Missing Performance History

# History indicator
validators["has_history"] = (
    validators["num_epochs_recorded"] > 1
).astype(int)

# Median CV from validators with actual history
cv_median = validators.loc[
    validators["has_history"] == 1,
    "efficiency_cv"
].median()

# Impute performance metrics
validators["median_efficiency_pct"] = (
    validators["median_efficiency_pct"]
    .fillna(0)
)

validators["performance_trend"] = (
    validators["performance_trend"]
    .fillna(0)
)

validators["efficiency_cv"] = (
    validators["efficiency_cv"]
    .fillna(cv_median)
)

candidate_features = [
    "commission",
    "log_lamports",
    "median_efficiency_pct",
    "efficiency_cv",
    "performance_trend",
    "num_epochs_recorded",
    "has_history"
]

In [ ]:
# Correlation Matrix / Heatmap of Candidate Features
corr_matrix = validators[candidate_features].corr()

corr_matrix

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f"
)

plt.title("Feature Correlation Matrix")
plt.show()

## **4) PCA and Dimensionality Reduction**

In [ ]:
# Scaling
from sklearn.preprocessing import StandardScaler

X = validators[candidate_features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled.shape

In [ ]:
# Fit PCA
from sklearn.decomposition import PCA
import pandas as pd

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

explained_var = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(candidate_features))],
    "Explained Variance": pca.explained_variance_ratio_,
    "Cumulative Variance": pca.explained_variance_ratio_.cumsum()
})

explained_var

In [ ]:
# Scree Plot (How many dimensions do we need?)
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(8,5))

plt.plot(
    range(1, len(candidate_features)+1),
    pca.explained_variance_ratio_,
    marker="o"
)

plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("Scree Plot")

plt.grid(True)

plt.show()

In [ ]:
# Cumulative Variance Plot
plt.figure(figsize=(8,5))

plt.plot(
    range(1, len(candidate_features)+1),
    np.cumsum(pca.explained_variance_ratio_),
    marker="o"
)

plt.axhline(0.80, linestyle="--")
plt.axhline(0.90, linestyle="--")

plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Variance Explained")

plt.grid(True)

plt.show()

In [ ]:
# PCA Loadings
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f"PC{i+1}" for i in range(len(candidate_features))],
    index=candidate_features
)

loadings

In [ ]:
loadings.style.background_gradient(cmap="coolwarm")

In [ ]:
# Project to First 2 PCs
pca_2 = PCA(n_components=2)

X_pca_2 = pca_2.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca_2,
    columns=["PC1", "PC2"]
)

plt.figure(figsize=(8,6))

plt.scatter(
    pca_df["PC1"],
    pca_df["PC2"],
    alpha=0.4,
    s=10
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Validators in PCA Space (PC1 vs PC2)")

plt.show()

### **PCA Findings**
#### *Keeping 4 Principal Components:*


1.   PC1: Validator Maturity / Scale
2.   Performance Quality
3.   Consistency / Trajectory
4.   Commision Policy



In [ ]:
# Final PCA Representation (4 PCs)

pca_final = PCA(n_components=4)

X_pca = pca_final.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2", "PC3", "PC4"]
)

# Keep validator identifier for later cluster analysis
pca_df["node_pubkey"] = validators["node_pubkey"].values

print(pca_df.shape)
display(pca_df.head())

pca_df.describe()

In [ ]:
# Pairwise PCA relationships

import seaborn as sns
import matplotlib.pyplot as plt

sns.pairplot(
    pca_df[["PC1", "PC2", "PC3", "PC4"]].sample(
        min(1000, len(pca_df)),
        random_state=42
    ),
    corner=True
)

plt.show()

In [ ]:
# Most extreme validators along each dimension

for pc in ["PC1", "PC2", "PC3", "PC4"]:
    print(f"\n{'='*60}")
    print(pc)

    print("\nTop 5:")
    display(
        pca_df.nlargest(5, pc)[
            ["node_pubkey", pc]
        ]
    )

    print("\nBottom 5:")
    display(
        pca_df.nsmallest(5, pc)[
            ["node_pubkey", pc]
        ]
    )

In [ ]:
pca_df.nlargest(20, "PC2")

## **5) Clustering**

### **5.1) Elbow Method**

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

inertia = []

K = range(1, 11)

for k in K:
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    km.fit(pca_df[["PC1","PC2","PC3","PC4"]])

    inertia.append(km.inertia_)

plt.figure(figsize=(8,5))
plt.plot(K, inertia, marker="o")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Within-Cluster Sum of Squares")
plt.title("Elbow Method")
plt.show()

### **5.1) Silhouette Analysis**

In [ ]:
from sklearn.metrics import silhouette_score

scores = []

for k in range(2,11):

    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = km.fit_predict(
        pca_df[["PC1","PC2","PC3","PC4"]]
    )

    score = silhouette_score(
        pca_df[["PC1","PC2","PC3","PC4"]],
        labels
    )

    scores.append(score)

plt.figure(figsize=(8,5))
plt.plot(range(2,11), scores, marker="o")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Analysis")
plt.show()

print(
    f"Best k = {range(2,11)[scores.index(max(scores))]}"
)

### K = 6
The elbow method showed substantial reductions in within-cluster variance up to approximately k=6, after which improvements diminished. Silhouette analysis similarly improved rapidly through k=6 before plateauing. Based on both criteria, a six-cluster solution was selected for subsequent analysis.

### **5.3) Final Model**

In [ ]:
kmeans = KMeans(
    n_clusters=6,
    random_state=42,
    n_init=10
)

pca_df["cluster"] = kmeans.fit_predict(
    pca_df[["PC1","PC2","PC3","PC4"]]
)

### **5.4) Interpretation**

In [ ]:
# Cluster Sizes
cluster_sizes = (
    pca_df["cluster"]
    .value_counts()
    .sort_index()
)

print(cluster_sizes)
print(cluster_sizes / len(pca_df))

In [ ]:
# Pairwise PCA plots colored by cluster

sns.pairplot(
    pca_df[["PC1", "PC2", "PC3", "PC4", "cluster"]],
    hue="cluster",
    palette="tab10",
    corner=True,
    plot_kws={"alpha": 0.6, "s": 12}
)

plt.suptitle("Pairwise PCA Space by Cluster", y=1.02)
plt.show()

In [ ]:
# Original Features
clustered = validators.copy()

clustered["cluster"] = pca_df["cluster"].values

cluster_profiles = (
    clustered
    .groupby("cluster")[candidate_features]
    .median()
)

display(cluster_profiles.round(3))

In [ ]:
cluster_profiles_mean = (
    clustered
    .groupby("cluster")[candidate_features]
    .mean()
)

display(cluster_profiles_mean.round(3))

In [ ]:
# Standardized Cluster Profile
cluster_zscores = (
    cluster_profiles
    - cluster_profiles.mean()
) / cluster_profiles.std()

display(cluster_zscores.round(2))

In [ ]:
# Heatmap
cluster_profiles = (
    clustered
    .groupby("cluster")[candidate_features]
    .median()
)

cluster_zscores = (
    cluster_profiles - cluster_profiles.mean()
) / cluster_profiles.std()

cluster_zscores = cluster_zscores.astype(float)

plt.figure(figsize=(10, 6))

sns.heatmap(
    cluster_zscores,
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f"
)

plt.title("Cluster Feature Profiles (Standardized)")
plt.show()